In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

bn = nn.BatchNorm1d(num_features=3)  # 3 features
x = torch.tensor([
    [1.0,  2.0,  3.0],
    [2.0,  4.0,  6.0],
    [3.0,  6.0,  9.0],
    [4.0,  8.0, 12.0],
])  # shape: [batch=4, features=3]

# ---- TRAIN MODE ----
bn.train()
y_train = bn(x)

print("TRAIN MODE")
print("output y_train:\n", y_train)
print("running_mean after forward:\n", bn.running_mean)
print("running_var after forward:\n", bn.running_var)

# ---- EVAL MODE ----
bn.eval()
y_eval = bn(x)

print("\nEVAL MODE")
print("output y_eval:\n", y_eval)
print("running_mean (should be unchanged):\n", bn.running_mean)
print("running_var  (should be unchanged):\n", bn.running_var)


TRAIN MODE
output y_train:
 tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]], grad_fn=<NativeBatchNormBackward0>)
running_mean after forward:
 tensor([0.2500, 0.5000, 0.7500])
running_var after forward:
 tensor([1.0667, 1.5667, 2.4000])

EVAL MODE
output y_eval:
 tensor([[0.7262, 1.1984, 1.4524],
        [1.6944, 2.7963, 3.3889],
        [2.6627, 4.3941, 5.3253],
        [3.6309, 5.9920, 7.2618]], grad_fn=<NativeBatchNormBackward0>)
running_mean (should be unchanged):
 tensor([0.2500, 0.5000, 0.7500])
running_var  (should be unchanged):
 tensor([1.0667, 1.5667, 2.4000])


In [6]:
bn = nn.BatchNorm1d(3); bn.train()
for i in range(5):
    x = torch.randn(4, 3) + i          # different batch mean each time
    _ = bn(x)                          # forward updates running_mean
    print(f"batch {i}: running_mean = {bn.running_mean.tolist()}")

bn.eval()
print(f"batch: running_mean = {bn.running_mean.tolist()}")

batch 0: running_mean = [0.0002709925174713135, 0.040883619338274, -0.030455023050308228]
batch 1: running_mean = [0.08705177903175354, 0.135629341006279, 0.06595619022846222]
batch 2: running_mean = [0.3525725305080414, 0.32639041543006897, 0.2980649769306183]
batch 3: running_mean = [0.6916632652282715, 0.621667742729187, 0.586015522480011]
batch 4: running_mean = [1.0232034921646118, 0.895012617111206, 0.8570636510848999]
batch: running_mean = [1.0232034921646118, 0.895012617111206, 0.8570636510848999]


In [8]:
import torch, torch.nn as nn

bn = nn.BatchNorm1d(3)

# 1) "Training" forwards: updates running stats (EMA)
bn.train()
for i in range(5):
    x = torch.randn(4, 3) + i
    _ = bn(x)
    print(f"batch {i}: running_mean = {bn.running_mean.tolist()}")

# 2) These are the EMA stats learned from those 5 batches
print("After training forwards (EMA stats):")
print("running_mean:", bn.running_mean.tolist())
print("running_var: ", bn.running_var.tolist())

# 3) Switch to eval: uses running stats, does NOT update them
bn.eval()
x_test = torch.randn(4, 3) + 100
_ = bn(x_test)

print("\nAfter eval forward (should be unchanged):")
print("running_mean:", bn.running_mean.tolist())
print("running_var: ", bn.running_var.tolist())


batch 0: running_mean = [-0.008804655633866787, 0.00947070587426424, -0.05257808044552803]
batch 1: running_mean = [0.06857971101999283, 0.21044722199440002, 0.14645346999168396]
batch 2: running_mean = [0.335129052400589, 0.39936962723731995, 0.30943775177001953]
batch 3: running_mean = [0.6113483905792236, 0.5717719793319702, 0.5956665873527527]
batch 4: running_mean = [0.9053171277046204, 0.9355052709579468, 1.0005508661270142]
After training forwards (EMA stats):
running_mean: [0.9053171277046204, 0.9355052709579468, 1.0005508661270142]
running_var:  [1.142888069152832, 1.2399674654006958, 1.0001814365386963]

After eval forward (should be unchanged):
running_mean: [0.9053171277046204, 0.9355052709579468, 1.0005508661270142]
running_var:  [1.142888069152832, 1.2399674654006958, 1.0001814365386963]


In [9]:
import torch, torch.nn as nn

bn = nn.BatchNorm1d(3)
bn.train()

for i in range(5):
    x = torch.randn(4, 3) + i

    batch_mu  = x.mean(dim=0)
    batch_var = x.var(dim=0, unbiased=False)  # matches BN's forward var

    _ = bn(x)  # updates running_mean/running_var in train mode

    print(f"\nBatch {i}")
    print("  batch_mu:       ", batch_mu.tolist())
    print("  batch_var:      ", batch_var.tolist())
    print("  running_mean:   ", bn.running_mean.tolist())
    print("  running_var:    ", bn.running_var.tolist())



Batch 0
  batch_mu:        [-1.0055296421051025, 0.2100130319595337, 0.437907338142395]
  batch_var:       [0.7006657719612122, 0.6742926836013794, 0.1431078314781189]
  running_mean:    [-0.10055296868085861, 0.021001305431127548, 0.04379073530435562]
  running_var:     [0.9934220910072327, 0.9899056553840637, 0.9190810322761536]

Batch 1
  batch_mu:        [0.6690859794616699, 1.0452762842178345, 1.3785967826843262]
  batch_var:       [0.396504282951355, 0.5787544846534729, 2.3528826236724854]
  running_mean:    [-0.02358907461166382, 0.12342880666255951, 0.17727133631706238]
  running_var:     [0.9469470977783203, 0.9680823683738708, 1.1408905982971191]

Batch 2
  batch_mu:        [0.9983247518539429, 1.0811703205108643, 2.1997647285461426]
  batch_var:       [0.051439110189676285, 0.2813431918621063, 0.1584346443414688]
  running_mean:    [0.07860230654478073, 0.21920296549797058, 0.3795206546783447]
  running_var:     [0.8591108918190002, 0.9087865352630615, 1.0479260683059692]

